In [17]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# Step 1: Importing libraries and seting the file path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

file_path = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet\raw_dataset\Housing_dataset\Housing-RTBESRI Rent Index\RIQ02_RTB_Rent_Index.xlsx'

In [19]:
#  Seeing which sheets are in the Excel file
excel_file = pd.ExcelFile(file_path)
print(excel_file.sheet_names)

['Data']


In [20]:
# Reading the Data sheet
rent = pd.read_excel(file_path, sheet_name='Data')

print(rent.shape)   # (rows, columns)
rent.head()

(376988, 5)


,Quarter,Number of Bedrooms,Property Type,Location,Value
0,2007Q4,All bedrooms,All property types,Carlow,805.30
1,2007Q4,All bedrooms,All property types,"Tullow, Carlow",724.37
2,2007Q4,All bedrooms,All property types,Cavan,588.23
3,2007Q4,All bedrooms,All property types,"Cootehill, Cavan",568.60
4,2007Q4,All bedrooms,All property types,Clare,716.65


In [21]:
# Renameing columns to short, simple names
print('Original names:', list(rent.columns))

rent.columns = ['quarter', 'bedrooms', 'property_type', 'location', 'avg_rent']

rent.head()

Original names: ['Quarter', 'Number of Bedrooms', 'Property Type', 'Location', 'Value']


,quarter,bedrooms,property_type,location,avg_rent
0,2007Q4,All bedrooms,All property types,Carlow,805.30
1,2007Q4,All bedrooms,All property types,"Tullow, Carlow",724.37
2,2007Q4,All bedrooms,All property types,Cavan,588.23
3,2007Q4,All bedrooms,All property types,"Cootehill, Cavan",568.60
4,2007Q4,All bedrooms,All property types,Clare,716.65


In [22]:
#Checking data types and missing values
print(rent.dtypes, '\n')
print('Missing values:')
print(rent.isnull().sum())

quarter              str
bedrooms             str
property_type        str
location             str
avg_rent         float64
dtype: object 

Missing values:
quarter          0
bedrooms         0
property_type    0
location         0
avg_rent         0
dtype: int64


In [23]:
# Looking at the categories
#Which bedroom groups and property types exist?
#How many locations are there?
print(rent['bedrooms'].value_counts(), '\n')
print(rent['property_type'].value_counts(), '\n')
print('Number of locations:', rent['location'].nunique())

bedrooms
All bedrooms     91408
1 to 3 bed       82506
1 to 2 bed       56111
Three bed        48547
Two bed          47101
Four plus bed    26201
One bed          25114
Name: count, dtype: int64 

property_type
All property types     141675
Apartment               89167
Semi detached house     57565
Terrace house           46038
Detached house          24851
Other flats             17692
Name: count, dtype: int64 

Number of locations: 437


In [24]:
#seeing Time period-- fixing it
#Quarters are written like `2021Q1`, the same format as the `quarter` column in our PPR data, so they can be joined later.
#We also split out the year and the quarter number.

print('First quarter:', rent['quarter'].min())
print('Last quarter: ', rent['quarter'].max())

rent['year'] = rent['quarter'].str[:4].astype(int)        # '2021Q1' -> 2021
rent['quarter_no'] = rent['quarter'].str[-1].astype(int)  # '2021Q1' -> 1

rent['year'].value_counts().sort_index()

First quarter: 2007Q4
Last quarter:  2025Q4


year
2007     2687
2008    17637
2009    22527
2010    22650
2011    22667
2012    22420
2013    23173
2014    23515
2015    23868
2016    23861
2017    23686
2018    23096
2019    22730
2020    22179
2021    20551
2022    17217
2023    16030
2024    14053
2025    12441
Name: count, dtype: int64

In [26]:
#Understanding the locations
#Locations are mixed:
#County: e.g. `Carlow`, `Cork`
#City / town / Dublin district: e.g. `Cork City`, `Donegal Town`, `Dublin 6`
#Smaller area inside a county: e.g. `Tullow, Carlow` (the part after the comma is the county)

#Let's look at the locations that have no comma
no_comma = []
for loc in rent['location'].unique():
    if ',' not in loc:
        no_comma.append(loc)

print(len(no_comma), 'locations without a comma:')
print(no_comma)

62 locations without a comma:
['Carlow', 'Cavan', 'Clare', 'Cork', 'Cork City', 'Donegal', 'Donegal Town', 'Dublin', 'Dublin 1', 'Dublin 2', 'Dublin 3', 'Dublin 4', 'Dublin 5', 'Dublin 6', 'Dublin 6W', 'Dublin 7', 'Dublin 8', 'Dublin 9', 'Dublin 10', 'Dublin 11', 'Dublin 12', 'Dublin 13', 'Dublin 14', 'Dublin 15', 'Dublin 16', 'Dublin 17', 'Dublin 18', 'Dublin 20', 'Dublin 24', 'Galway', 'Galway City', 'Kerry', 'Kildare', 'Kildare Town', 'Kilkenny', 'Kilkenny City', 'Laois', 'Leitrim', 'Limerick', 'Limerick City', 'Longford', 'Louth', 'Mayo', 'Meath', 'Monaghan', 'Offaly', 'Roscommon', 'Sligo', 'Tipperary', 'Tipperary Town', 'Waterford', 'Waterford City', 'Westmeath', 'Wexford', 'Wicklow', 'Wicklow Town', 'Cavan Town', 'Carlow Town', 'Longford Town', 'Roscommon Town', 'Sligo Town', 'Monaghan Town']


In [27]:
#Adding a county column
# If the location has a comma, e.g. `Tullow, Carlow` -> county = `Carlow` (the part after the comma)
#If not, e.g. `Cork City`, `Dublin 6`, `Donegal Town` -> county = the **first word**

#We also add `location_level`:
# `county` = the row is for the whole county (location is exactly the county name)
# `local` = the row is for a smaller area inside the county

county_list = []
for loc in rent['location']:
    if ',' in loc:
        county_list.append(loc.split(',')[1].strip())   # part after the comma
    else:
        county_list.append(loc.split(' ')[0])           # first word

rent['county'] = county_list

rent['location_level'] = 'local'
rent.loc[rent['location'] == rent['county'], 'location_level'] = 'county'

print(rent['county'].nunique(), 'counties')
print(rent['county'].value_counts().sort_index())
print(rent['location_level'].value_counts())

48 counties
county
Carlow        4985
Cavan         4776
Clare         5954
Cork         32627
Donegal       6583
Dublin       54909
Dublin 1      2556
Dublin 10     1196
Dublin 11     3165
Dublin 12     3960
Dublin 13     3159
Dublin 14     4180
Dublin 15     9575
Dublin 16     4003
Dublin 17      830
Dublin 18     4818
Dublin 2      3241
Dublin 20     1254
Dublin 22     1478
Dublin 24     3473
Dublin 3      6642
Dublin 4      7440
Dublin 5      2524
Dublin 6      8316
Dublin 6W     2959
Dublin 7      5250
Dublin 8      7477
Dublin 9      6378
Galway       20430
Kerry         8533
Kildare      16440
Kilkenny      4397
Laois         5632
Leitrim       2739
Limerick     11603
Longford      4449
Louth         6965
Mayo          9732
Meath        12441
Monaghan      4616
Offaly        6148
Roscommon     5371
Sligo         5362
Tipperary    11338
Waterford    10238
Westmeath     7274
Wexford       8841
Wicklow      10731
Name: count, dtype: int64
location_level
local     315005
county     

In [28]:
#Keeping only the study period (2021 onwards)
#The rent data ends in **2025Q4**, so 2021–2025 is fully covered.

rent_all_years = rent.copy()   # keep a copy of all years, just in case

rent = rent[rent['year'] >= 2021]

print('Rows from 2021 onwards:', len(rent))

Rows from 2021 onwards: 80292
